# 小米 14：初版与最新版的无频率门槛对照

## tl;dr

成功进程 48/48，全部场景完整性及稳定输出校验通过：False。两版均跳过频率起跑门槛；以下为独立场次的描述性结果，不是已验收的严格回归基线。

## Context & Methods

### Key Assumptions

- 小米 14 / Snapdragon 8 Gen 3 / Android 16；v0.1.0 与本地 v0.3.0。
- 固定八场景 × 两版本 × 三轮，AB / BA / AB，每进程三遍，结果缓存关闭。
- 两版均不等待频率恢复；保留前台检查、频率、温度及 8 秒进程间隔，不关闭系统温控。
- 每格取三进程中位数；首次含服务与模型创建、加载及首遍翻译，热态为每进程后两遍中位数；内存为首遍 native RSS。
- 默认参数收益：初版 Async 1 / batch 1024，对比新版 Blocking / batch 512 + 英文前缀表；不是纯内核收益。
- 缺失或失败格不计算中位数，波动阈值仍标注 10%，不拼接别的场次。
- 输出哈希可揭示文本变化，但没有 COMET 评分就不宣称质量不变。

## Data

原始数据 `native-results.json`；采集器、检查器、清单和测试入口快照位于 `collector/`。源码、二进制、模型及语料摘要均在原始记录内。复核仅需标准库；执行本 notebook 另需 nbformat、nbclient、ipykernel、IPython。

In [1]:
from pathlib import Path
import json, runpy
from IPython.display import Markdown, display
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'tools/version-bench/check.py').is_file())
record = root / 'benchmarks/v0.3.0/mi14-2026-09-09/initial-to-current/ungated'
checks = runpy.run_path(str(record / 'verify.py'))
result = checks['analyze']()

## Results

### 1. 完整性与证据边界

In [2]:
print({k:v for k,v in result.items() if k not in ('rows', 'same_config', 'defaults')})
assert not result['accepted_as_strict_baseline']
assert result['strict_checker_exit_code'] == 2
print('Incomplete cells:', [r for r in result['rows'] if not r['complete']])

{'measured_at': '2026-09-09T20:48:19+0800', 'accepted_as_strict_baseline': False, 'strict_checker_exit_code': 2, 'strict_checker_diagnostic': 'INVALID: exploratory runs cannot be a complete version baseline', 'collection_complete': True, 'complete': False, 'recorded_processes': 48, 'planned_processes': 48, 'successful_processes': 48, 'starting_temperature_range_c': [33.2, 37.0], 'starting_frequency_caps_khz': {'2': [2188800, 2323200], '7': [1824000, 1939200]}}
Incomplete cells: [{'scenario': 'enzh_b512p', 'version': 'v0.1.0', 'successful_processes': 3, 'complete': False, 'errors': [], 'validation_error': 'v0.1.0/enzh_b512p: output unstable within a version'}, {'scenario': 'pivot_b512p', 'version': 'v0.1.0', 'successful_processes': 3, 'complete': False, 'errors': [], 'validation_error': 'v0.1.0/pivot_b512p: output unstable within a version'}]


### 2. 固定八场景、两版本；速度与内存

In [3]:
table = ['| 场景 | 版本 | 首次 条/秒 | 热态 条/秒 | 峰值 RSS MiB |', '|---|---|---:|---:|---:|']
for r in result['rows']:
    if r['complete']:
        table.append(f"| {r['scenario']} | {r['version']} | {r['cold_inputs_per_second']:.2f} | {r['warm_inputs_per_second']:.2f} | {r['peak_rss_mib']:.2f} |")
    else:
        table.append(f"| {r['scenario']} | {r['version']} | 未通过校验 | 未通过校验 | 未通过校验 |")
display(Markdown('\n'.join(table)))

| 场景 | 版本 | 首次 条/秒 | 热态 条/秒 | 峰值 RSS MiB |
|---|---|---:|---:|---:|
| enzh_w1 | v0.1.0 | 27.97 | 30.72 | 348.49 |
| enzh_w1 | v0.3.0 | 51.38 | 58.81 | 207.40 |
| enzh_w2 | v0.1.0 | 51.26 | 58.46 | 540.06 |
| enzh_w2 | v0.3.0 | 94.40 | 109.70 | 302.86 |
| enzh_w4 | v0.1.0 | 78.32 | 98.43 | 940.46 |
| enzh_w4 | v0.3.0 | 146.92 | 179.53 | 522.68 |
| pivot_w1 | v0.1.0 | 12.79 | 14.04 | 530.37 |
| pivot_w1 | v0.3.0 | 24.75 | 26.86 | 311.78 |
| enzh_b512p | v0.1.0 | 未通过校验 | 未通过校验 | 未通过校验 |
| enzh_b512p | v0.3.0 | 56.61 | 60.67 | 207.07 |
| pivot_b512p | v0.1.0 | 未通过校验 | 未通过校验 | 未通过校验 |
| pivot_b512p | v0.3.0 | 27.11 | 29.18 | 308.46 |
| enzh_w2_512p | v0.1.0 | 50.10 | 59.74 | 532.11 |
| enzh_w2_512p | v0.3.0 | 93.80 | 106.34 | 284.43 |
| pivot_w2_512p | v0.1.0 | 23.76 | 26.18 | 759.06 |
| pivot_w2_512p | v0.3.0 | 44.44 | 50.09 | 436.24 |

### 3. 波动、频率条件及同配置差异

In [4]:
print('Spreads above 10%:')
for r in result['rows']:
    for metric, spread in r.get('spread_pct', {}).items():
        if spread > 10:
            print(r['scenario'], r['version'], metric, round(spread, 2), r['raw_values'][metric])
print(json.dumps(result['same_config'], ensure_ascii=False, indent=2))

Spreads above 10%:
enzh_w2_512p v0.3.0 cold_ms 13.1 [1974.193, 2132.256, 2253.571]
enzh_w2_512p v0.3.0 warm_ms 11.38 [1688.2115, 1880.687, 1902.147]
[
  {
    "old_scenario": "enzh_w1",
    "new_scenario": "enzh_w1",
    "complete": true,
    "old_cold_inputs_per_second": 27.969551786534144,
    "new_cold_inputs_per_second": 51.37930308085715,
    "cold_speedup": 1.836972700634894,
    "cold_speed_change_pct": 83.6972700634894,
    "warm_speedup": 1.9144214230780938,
    "old_peak_rss_mib": 348.492,
    "new_peak_rss_mib": 207.395,
    "rss_reduction_pct": 40.48787346624887,
    "per_round_cold_speedups": [
      1.8136475467304611,
      1.836972700634894,
      1.8253890987089123
    ],
    "all_starting_caps_equal": false,
    "starting_temperature_range_c": [
      33.2,
      36.6
    ],
    "spreads_above_10pct": [],
    "stable_output_hashes_equal": null
  },
  {
    "old_scenario": "enzh_w2",
    "new_scenario": "enzh_w2",
    "complete": true,
    "old_cold_inputs_per_second":

### 4. 各版默认单线程路径对照

In [5]:
print(json.dumps(result['defaults'], ensure_ascii=False, indent=2))

[
  {
    "old_scenario": "enzh_w1",
    "new_scenario": "enzh_b512p",
    "complete": true,
    "old_cold_inputs_per_second": 27.969551786534144,
    "new_cold_inputs_per_second": 56.61187015048567,
    "cold_speedup": 2.024053534448889,
    "cold_speed_change_pct": 102.4053534448889,
    "warm_speedup": 1.9749901185381078,
    "old_peak_rss_mib": 348.492,
    "new_peak_rss_mib": 207.074,
    "rss_reduction_pct": 40.57998461944607,
    "per_round_cold_speedups": [
      2.0034223759315384,
      2.024053534448889,
      2.0285633139369925
    ],
    "all_starting_caps_equal": false,
    "starting_temperature_range_c": [
      33.2,
      36.8
    ],
    "spreads_above_10pct": [],
    "stable_output_hashes_equal": null
  },
  {
    "old_scenario": "pivot_w1",
    "new_scenario": "pivot_b512p",
    "complete": true,
    "old_cold_inputs_per_second": 12.791346909642565,
    "new_cold_inputs_per_second": 27.1074335359614,
    "cold_speedup": 2.1192008728593597,
    "cold_speed_change_pct"

## Takeaways

使用中位数并不能消除系统调频影响。以上倍率只描述此设备、此语料、此场次；默认参数对照包含分句与组批变化。不能将它们与下午的高频场次拼接，也不能用本记录建立严格自动回归基线。